In [ ]:
import torch
import clip
from PIL import Image

# 加载CLIP模型
print("正在加载CLIP模型...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)
print(f"使用设备: {device}，模型加载完成")

# 加载图片
image_path = r"D:\badou\第10周：多模态大模型\小狗.jpg"
image = Image.open(image_path)
image_input = preprocess(image).unsqueeze(0).to(device)

# 定义候选类别
candidate_labels = ["dog", "cat", "bird", "golden retriever", "puppy", "husky", "wolf", "fox"]

# 编码文本
text_inputs = clip.tokenize(candidate_labels).to(device)

# Zero-Shot分类
with torch.no_grad():
    image_features = model.encode_image(image_input)
    text_features = model.encode_text(text_inputs)
    
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    
    similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)

# 输出结果
print("\n分类结果:")
values, indices = similarity[0].topk(5)

for i, (idx, value) in enumerate(zip(indices, values)):
    print(f"{i+1}. {candidate_labels[idx]}: {value.item():.2%}")

print(f"\n最佳预测: {candidate_labels[indices[0]]}")